In [1]:
import os
import pandas as pd
# use polars when using very large dataframes as polars is faster than pandas
import polars as pl

# use pickle to save and load data objects (when saving a df using pickle, you save more information)
import pickle
import matplotlib.pyplot as plt


# With the ic() method you can print something that is between the brackets and automatically also print its name
from icecream import ic


## Load the history_per_year parquet dataset and show the column names

In [ ]:
# Read the Parquet file using Pandas
# df = pd.read_parquet("C:/Users/jasmi/Documents/2025_Data_analyse/2025_03_Group_project_Supermarket/Data/data/history-per-year.parquet")

# print("The columns names are:", df.columns)
# print(f'the number of columns is {len(df.columns)}')


# Read the Parquet file
df_polars = pl.scan_parquet("C:/Users/jasmi/Documents/2025_Data_analyse/2025_03_Group_project_Supermarket/Data/data/combined_history_items_stores.parquet")

# Collect the DataFrame
df_polars_collected = df_polars.collect()

# Display column names and their data types
print("Column Names and Data Types are:")
for column in df_polars_collected.columns:
 print(f"{column}: {df_polars_collected[column].dtype}")


Column Names and Data Types are:
id: Int32
store_nbr: Int8
item_nbr: Float32
unit_sales: Float32
onpromotion: Boolean
day: Int8
month: Categorical(ordering='physical')
family: Categorical(ordering='physical')
class: Int16
perishable: Int8
city: Categorical(ordering='physical')
state: Categorical(ordering='physical')
type: Categorical(ordering='physical')
cluster: Int8


## Show the tops rows of the history-per-year dataframe and print the number of rows

In [3]:
# Calculate the number of observations in millions
nr_observations_million = df_polars_collected.shape[0] / 1000000
print(f'The number of observations is {nr_observations_million:.2f} million')

df_polars.head(10).collect()


The number of observations is 125.50 million


id,store_nbr,item_nbr,unit_sales,onpromotion,day,month,family,class,perishable,city,state,type,cluster
i32,i8,f32,f32,bool,i8,cat,cat,i16,i8,cat,cat,cat,i8
0,25,103665.0,7.0,null,1,"""2013-01""","""BREAD/BAKERY""",2712,1,"""Salinas""","""Santa Elena""","""D""",1
1,25,105574.0,1.0,null,1,"""2013-01""","""GROCERY I""",1045,0,"""Salinas""","""Santa Elena""","""D""",1
2,25,105575.0,2.0,null,1,"""2013-01""","""GROCERY I""",1045,0,"""Salinas""","""Santa Elena""","""D""",1
3,25,108079.0,1.0,null,1,"""2013-01""","""GROCERY I""",1030,0,"""Salinas""","""Santa Elena""","""D""",1
4,25,108701.0,1.0,null,1,"""2013-01""","""DELI""",2644,1,"""Salinas""","""Santa Elena""","""D""",1
5,25,108786.0,3.0,null,1,"""2013-01""","""CLEANING""",3044,0,"""Salinas""","""Santa Elena""","""D""",1
6,25,108797.0,1.0,null,1,"""2013-01""","""GROCERY I""",1004,0,"""Salinas""","""Santa Elena""","""D""",1
7,25,108952.0,1.0,null,1,"""2013-01""","""CLEANING""",3024,0,"""Salinas""","""Santa Elena""","""D""",1
8,25,111397.0,13.0,null,1,"""2013-01""","""GROCERY I""",1072,0,"""Salinas""","""Santa Elena""","""D""",1


## Print the number of store_nbr values, states and cities

In [ ]:
# Collect the DataFrame
df_polars_collected = df_polars.collect()

# Identify the number of unique values in the store_nbr column
unique_store_nbr_count = df_polars_collected['store_nbr'].n_unique()

# Identify the number of unique values in the state column
unique_state_count = df_polars_collected['state'].n_unique()

# Identify the number of unique values in the city column
unique_city_count = df_polars_collected['city'].n_unique()

# Identify the number of unique values in the cluster column
unique_cluster_count = df_polars_collected['cluster'].n_unique()


print(f"The number of unique values in the store_nbr column is: {unique_store_nbr_count}")
print(f"The number of unique values in the state column is: {unique_state_count}")
print(f"The number of unique values in the city column is: {unique_city_count}")
print(f"The number of unique values in the cluster column is: {unique_cluster_count}")

## Groupe by state

In [ ]:

# Read the Parquet file into a Polars DataFrame
# df_polars = pl.read_parquet("C:/Users/jasmi/Documents/2025_Data_analyse/2025_03_Group_project_Supermarket/Data/data/combined_history_items_stores.parquet")

# Convert the Polars DataFrame to a pandas DataFrame
# df_pandas = df_polars.to_pandas()

# Group by state and sum the sales
# sales_by_state = df_pandas.groupby('state')['unit_sales'].sum().reset_index()

# Sort by 'unit_sales' descending
# sales_by_state_sorted = sales_by_state.sort_values(by='unit_sales', ascending=False)

# print(sales_by_state_sorted)


: 

# Group by Store_nbr

In [ ]:
import polars as pl

# Read the Parquet file into a Polars DataFrame
df_polars = pl.read_parquet("C:/Users/jasmi/Documents/2025_Data_analyse/2025_03_Group_project_Supermarket/Data/data/combined_history_items_stores.parquet")

# Convert the Polars DataFrame to a pandas DataFrame
df_pandas = df_polars.to_pandas()

# Group by store_nbr and sum the sales
sales_by_store_nbr = df_pandas.groupby('store_nbr')['unit_sales'].sum().reset_index()

# Sort by 'unit_sales' descending
sales_by_store_nbr_sorted = sales_by_store_nbr.sort_values(by='unit_sales', ascending=False)
total_unit_sales = sales_by_store_nbr_sorted['unit_sales'].sum()

sales_by_store_nbr_sorted['proportion to to total sales'] = sales_by_store_nbr_sorted['unit_sales']/total_unit_sales

sales_by_store_nbr_sorted['accumulative proportion'] = sales_by_store_nbr_sorted['proportion to to total sales'].cumsum()

print(sales_by_store_nbr_sorted)


## Scope the original dataframe based on store_nbr that together represent about 50% or 70% of the total sales

In [ ]:

# Filter the DataFrame to get store_nbr where accumulative proportion is about 70%
store_nbr_70_selection = sales_by_store_nbr_sorted[sales_by_store_nbr_sorted['accumulative proportion'] <= 0.70]['store_nbr'].values
store_nbr_50_selection = sales_by_store_nbr_sorted[sales_by_store_nbr_sorted['accumulative proportion'] <= 0.50]['store_nbr'].values

df_70pcnt_scoped = df_pandas[df_pandas['store_nbr'].isin(store_nbr_70_selection)]
df_50pcnt_scoped = df_pandas[df_pandas['store_nbr'].isin(store_nbr_50_selection)]
print(df_70pcnt_scoped.head(10))
print(df_50pcnt_scoped.head(10))

nr_observations_70_scoped_million = df_70pcnt_scoped.shape[0] / 1000000
nr_observations_50_scoped_million = df_50pcnt_scoped.shape[0] / 1000000
print(f'The number of observations in the 70%_scoped dataframe is {nr_observations_70_scoped_million:.2f} million')
print(f'The number of observations in the 50%_scoped dataframe is {nr_observations_50_scoped_million:.2f} million')

        id  store_nbr  item_nbr  unit_sales onpromotion  day    month  \
1596  1596          2  103665.0         5.0        None    2  2013-01   
1597  1597          2  105574.0        21.0        None    2  2013-01   
1598  1598          2  105577.0         6.0        None    2  2013-01   
1599  1599          2  105693.0         4.0        None    2  2013-01   
1600  1600          2  105737.0         9.0        None    2  2013-01   
1601  1601          2  105857.0         5.0        None    2  2013-01   
1602  1602          2  106716.0        10.0        None    2  2013-01   
1603  1603          2  108079.0         1.0        None    2  2013-01   
1604  1604          2  108696.0         5.0        None    2  2013-01   
1605  1605          2  108698.0         6.0        None    2  2013-01   

            family  class  perishable   city      state type  cluster  
1596  BREAD/BAKERY   2712           1  Quito  Pichincha    D       13  
1597     GROCERY I   1045           0  Quito  Pichin

## Scope the original dataframe down to the top 10 best selling store

In [ ]:
df_top_10 = df_pandas[df_pandas['store_nbr'].isin([44, 45, 47, 3, 49, 46, 48, 51, 8, 50])]
print(df_top_10.head(10))

nr_observations_top_10_million = df_top_10.shape[0] / 1000000
print(f'The number of observations in the top 10 scoped dataframe is {nr_observations_top_10_million:.2f} million')

        id  store_nbr  item_nbr  unit_sales onpromotion  day    month  \
2699  2699          3  103665.0         6.0        None    2  2013-01   
2700  2700          3  105574.0        36.0        None    2  2013-01   
2701  2701          3  105575.0        21.0        None    2  2013-01   
2702  2702          3  105577.0         6.0        None    2  2013-01   
2703  2703          3  105693.0         2.0        None    2  2013-01   
2704  2704          3  105737.0        12.0        None    2  2013-01   
2705  2705          3  105857.0        36.0        None    2  2013-01   
2706  2706          3  106716.0        11.0        None    2  2013-01   
2707  2707          3  108079.0         2.0        None    2  2013-01   
2708  2708          3  108696.0         9.0        None    2  2013-01   

            family  class  perishable   city      state type  cluster  
2699  BREAD/BAKERY   2712           1  Quito  Pichincha    D        8  
2700     GROCERY I   1045           0  Quito  Pichin

## Save object files with pickle

In [ ]:
# Create dictionary 'dc_scoped_df' with objects that will be used in the next code parts.
dc_scoped_df = {
    'df_pl_orig':      df_polars,
    'df_pd_pandas':   df_pandas,
    'df_70pcnt_scoped':  df_70pcnt_scoped,
    'df_50pcnt_scoped':  df_50pcnt_scoped,
    'df_top_10':  df_top_10
}

# Save dc_scoped_df as 'dc-scoped-df.pkl'
with open('../Data/data/dc-scoped-df.pkl', 'wb') as pickle_file:
    pickle.dump(dc_scoped_df, pickle_file)